In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    IntegerType,
    DoubleType,
    DecimalType,
    DateType,
    TimestampType,
    BooleanType
)

# ============================================
# CONFIGURACIÓN DEL PROYECTO
# ============================================

CATALOG = "banco"
SCHEMA = "banking_analytics"

BRONZE_TABLES = {
    "branches": f"{CATALOG}.{SCHEMA}.bronze_branches",
    "customers": f"{CATALOG}.{SCHEMA}.bronze_customers",
    "accounts": f"{CATALOG}.{SCHEMA}.bronze_accounts",
    "transactions": f"{CATALOG}.{SCHEMA}.bronze_transactions"
}

SILVER_TABLES = {
    "branches": f"{CATALOG}.{SCHEMA}.silver_branches",
    "customers": f"{CATALOG}.{SCHEMA}.silver_customers",
    "accounts": f"{CATALOG}.{SCHEMA}.silver_accounts",
    "transactions": f"{CATALOG}.{SCHEMA}.silver_transactions"
}

print("Configuración Silver completada.")

In [0]:
# ============================================
# INSPECCIÓN DE TABLAS BRONZE
# ============================================

for entity, table_name in BRONZE_TABLES.items():

    print(f"\n{'=' * 60}")
    print(f"ENTIDAD: {entity.upper()}")
    print(f"TABLA: {table_name}")
    print(f"{'=' * 60}")

    df = spark.table(table_name)

    print(f"Total registros: {df.count()}")

    df.printSchema()

In [0]:
for entity, table_name in BRONZE_TABLES.items():

    print(f"\nMuestra de {entity.upper()}")

    df = spark.table(table_name)

    display(df.limit(5))

In [0]:
#########################################################
# REGLAS DE CALIDAD Definiendo tipo de datos
#########################################################
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Leer tabla Bronze
df_branches = spark.table(BRONZE_TABLES["branches"])

# Convertir y estandarizar atributos
df_branches_prepared = (
    df_branches

    .withColumn(
        "branch_id",
        F.upper(F.trim(F.col("branch_id")))
    )

    .withColumn(
        "branch_name",
        F.initcap(F.trim(F.col("branch_name")))
    )

    .withColumn(
        "city",
        F.initcap(F.trim(F.col("city")))
    )

    .withColumn(
        "region",
        F.initcap(F.trim(F.col("region")))
    )

    .withColumn(
        "country",
        F.upper(F.trim(F.col("country")))
    )

    # Conversión segura de String a Date
    .withColumn(
        "opening_date",
        F.expr("try_cast(opening_date AS DATE)")#se intenta convertir la columna en date, pero si no se logra lo vuelve NULL
    )

    # Conversión de String a Boolean
    .withColumn(
        "is_active",
        F.when(
            F.lower(F.trim(F.col("is_active"))).isin(
                "true", "1", "yes"
            ),
            F.lit(True)
        )
        .when(
            F.lower(F.trim(F.col("is_active"))).isin(
                "false", "0", "no"
            ),
            F.lit(False)
        )
        .otherwise(F.lit(None).cast("boolean"))
    )
)

display(df_branches_prepared.limit(10))

df_branches_prepared.printSchema()

In [0]:
# Definir ventana para identificar duplicados
window_branch = (
    Window
    .partitionBy("branch_id")
    .orderBy(
        F.col("_ingestion_timestamp").desc_nulls_last(),
        F.col("_source_file").asc()
    )
)

df_branches_prepared = (
    df_branches_prepared
    .withColumn(
        "_row_number",
        F.row_number().over(window_branch)
    )
    .withColumn(
        "_is_duplicate",
        (F.col("branch_id").isNotNull()) &
        (F.col("branch_id") != "") &
        (F.col("_row_number") > 1)
    )
)

display(
    df_branches_prepared
    .filter(F.col("_is_duplicate"))
)

In [0]:
df_branches_quality = (
    df_branches_prepared

    .withColumn(
        "_rejection_reason",
        F.concat_ws(
            ", ",

            F.when(
                F.col("branch_id").isNull() |
                (F.col("branch_id") == ""),
                F.lit("MISSING_BRANCH_ID")
            ),

            F.when(
                F.col("branch_name").isNull() |
                (F.trim(F.col("branch_name")) == ""),
                F.lit("MISSING_BRANCH_NAME")
            ),

            F.when(
                F.col("city").isNull() |
                (F.trim(F.col("city")) == ""),
                F.lit("MISSING_CITY")
            ),

            F.when(
                F.col("region").isNull() |
                (F.trim(F.col("region")) == ""),
                F.lit("MISSING_REGION")
            ),

            F.when(
                F.col("country").isNull() |
                (F.trim(F.col("country")) == ""),
                F.lit("MISSING_COUNTRY")
            ),

            F.when(
                F.col("opening_date").isNull(),
                F.lit("INVALID_OPENING_DATE")
            ),

            F.when(
                F.col("is_active").isNull(),
                F.lit("INVALID_IS_ACTIVE")
            ),

            F.when(
                F.col("_is_duplicate"),
                F.lit("DUPLICATE_BRANCH_ID")
            )
        )
    )
)

display(
    df_branches_quality
    .select("branch_id", "_rejection_reason")
)

In [0]:
# ============================================
# SEPARACIÓN DE REGISTROS (VALIDOS y RECHAZADOS)
# ============================================

df_silver_branches = (
    df_branches_quality
    .filter(F.col("_rejection_reason") == "")
    .drop(
        "_row_number",
        "_is_duplicate",
        "_rejection_reason"
    )
)

df_quarantine_branches = (
    df_branches_quality
    .filter(F.col("_rejection_reason") != "")
    .drop("_row_number")
)

print("Registros válidos:", df_silver_branches.count())
print("Registros rechazados:", df_quarantine_branches.count())

In [0]:
# ============================================
# PERSISTENCIA SILVER
# ============================================

(
    df_silver_branches.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLES["branches"])
)

# ============================================
# PERSISTENCIA QUARANTINE
# ============================================

(
    df_quarantine_branches.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.{SCHEMA}.quarantine_branches"
    )
)

print("Tablas guardadas correctamente.")

In [0]:
# Consultar tabla Silver
df_silver_result = spark.table(
    "banco.banking_analytics.silver_branches"
)

# Consultar tabla Quarantine
df_quarantine_result = spark.table(
    "banco.banking_analytics.quarantine_branches"
)

print("========== RESULTADOS BRANCHES ==========")

print("Bronze:", df_branches.count())

print("Silver:", df_silver_result.count())

print("Quarantine:", df_quarantine_result.count())

print(
    "Total procesado:",
    df_silver_result.count() +
    df_quarantine_result.count()
)

df_silver_result.printSchema()

display(df_silver_result)

In [0]:
#####################################################
## LEER BRONZE CUSTOMER 
#####################################################
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# Leer tabla Bronze
df_customers = spark.table(BRONZE_TABLES["customers"])

print("Total registros Bronze:", df_customers.count())



In [0]:
#######################################################
# VERSION NORMALIZADA
#######################################################

df_customers_base = (
    df_customers

    # Normalización de identificadores y textos
    .withColumn(
        "customer_id",
        F.when(
            F.trim(F.col("customer_id")) == "",
            None
        ).otherwise(F.upper(F.trim(F.col("customer_id"))))
    )
    .withColumn(
        "customer_name",
        F.when(
            F.trim(F.col("customer_name")) == "",
            None
        ).otherwise(F.initcap(F.trim(F.col("customer_name"))))
    )
    .withColumn(
        "country",
        F.when(
            F.trim(F.col("country")) == "",
            None
        ).otherwise(F.upper(F.trim(F.col("country"))))
    )
    .withColumn(
        "email",
        F.when(
            F.trim(F.col("email")) == "",
            None
        ).otherwise(F.lower(F.trim(F.col("email"))))
    )
    .withColumn(
        "customer_segment",
        F.upper(F.trim(F.col("customer_segment")))
    )

    # Conversión de fechas
    .withColumn(
        "birth_date",
        F.expr("try_cast(birth_date AS DATE)")
    )
    .withColumn(
        "registration_date",
        F.expr("try_cast(registration_date AS DATE)")
    )
)

display(df_customers_base)


In [0]:
# Revisar el esquema y los datos que tenemos


df_customers_base.printSchema()

display(
    df_customers_base.select(
        "customer_id",
        "customer_name",
        "birth_date",
        "registration_date",
        "email",
        "country",
        "customer_segment"
    ).limit(10)
)


In [0]:

# IDENTIFCACIÓN DE DUPLICADOS CUSTOMERS
window_customer = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("_ingestion_timestamp").desc_nulls_last(),
        F.col("_source_file").asc()
    )
)

df_customers_dedup = (
    df_customers_base
    .withColumn(
        "_row_number",
        F.row_number().over(window_customer)
    )
    .withColumn(
        "_is_duplicate",
        F.when(
            F.col("customer_id").isNotNull()
            & (F.col("_row_number") > 1),
            True
        ).otherwise(False)
    )
)

print("Registros después de identificar duplicados:", df_customers_dedup.count())

display(
    df_customers_dedup
    .filter(F.col("_is_duplicate") == True)
)


In [0]:
################################################################
# APLICACNDO REGLAS DE CALIDAD
################################################################

valid_segments = [
    "STANDARD",
    "GOLD",
    "PLATINUM",
    "PREMIUM"
]

email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

df_customers_quality = (
    df_customers_dedup
    .withColumn(
        "_rejection_reason",
        F.concat_ws(
            ", ",

            F.when(
                F.col("customer_id").isNull(),
                "MISSING_CUSTOMER_ID"
            ),

            F.when(
                F.col("customer_name").isNull(),
                "MISSING_CUSTOMER_NAME"
            ),

            F.when(
                F.col("birth_date").isNull()
                | (F.col("birth_date") > F.current_date()),
                "INVALID_BIRTH_DATE"
            ),

            F.when(
                F.col("registration_date").isNull()
                | (F.col("registration_date") > F.current_date()),
                "INVALID_REGISTRATION_DATE"
            ),

            F.when(
                F.col("email").isNotNull()
                & (~F.col("email").rlike(email_pattern)),
                "INVALID_EMAIL"
            ),

            F.when(
                F.col("country").isNull(),
                "MISSING_COUNTRY"
            ),

            F.when(
                ~F.col("customer_segment").isin(valid_segments)
                | F.col("customer_segment").isNull(),
                "INVALID_CUSTOMER_SEGMENT"
            ),

            F.when(
                F.col("_is_duplicate"),
                "DUPLICATE_CUSTOMER_ID"
            )
        )
    )
)

display(
    df_customers_quality
    .select(
        "customer_id",
        "customer_name",
        "email",
        "customer_segment",
        "_is_duplicate",
        "_rejection_reason"
    )
    .filter(F.col("_rejection_reason") != "")
)


In [0]:

# Revision de resultados

total_bronze = df_customers.count()

total_silver = (
    df_customers_quality
    .filter(F.col("_rejection_reason") == "")
    .count()
)

total_quarantine = (
    df_customers_quality
    .filter(F.col("_rejection_reason") != "")
    .count()
)

print("Total Bronze:", total_bronze)
print("Total Silver esperado:", total_silver)
print("Total Quarantine esperado:", total_quarantine)
print("Total procesado:", total_silver + total_quarantine)

display(
    df_customers_quality
    .filter(F.col("_rejection_reason") != "")
    .groupBy("_rejection_reason")
    .count()
    .orderBy(F.desc("count"))
)


In [0]:

# DataFrame Silver: registros válidos
df_silver_customers = (
    df_customers_quality
    .filter(F.col("_rejection_reason") == "")
    .drop(
        "_row_number",
        "_is_duplicate",
        "_rejection_reason"
    )
)

# DataFrame Quarantine: registros rechazados
df_quarantine_customers = (
    df_customers_quality
    .filter(F.col("_rejection_reason") != "")
    .drop("_row_number")
)

# Guardar tabla Silver
(
    df_silver_customers.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLES["customers"])
)

# Guardar tabla Quarantine
(
    df_quarantine_customers.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.{SCHEMA}.quarantine_customers"
    )
)

print("Tablas Customers creadas correctamente.")


In [0]:

# Validaciones de tablas delta
df_silver_saved = spark.table(SILVER_TABLES["customers"])

df_quarantine_saved = spark.table(
    f"{CATALOG}.{SCHEMA}.quarantine_customers"
)

print("Bronze:", df_customers.count())
print("Silver:", df_silver_saved.count())
print("Quarantine:", df_quarantine_saved.count())

print(
    "Total reconciliado:",
    df_silver_saved.count() + df_quarantine_saved.count()
)


In [0]:

# Leer tabla Bronze de Accounts
df_accounts = spark.table(BRONZE_TABLES["accounts"])

# Leer tablas Silver de referencia
df_customers_reference = spark.table(SILVER_TABLES["customers"])

df_branches_reference = spark.table(SILVER_TABLES["branches"])

# Revisar cantidad de registros
print("Total Accounts Bronze:", df_accounts.count())
print("Total Customers Silver:", df_customers_reference.count())
print("Total Branches Silver:", df_branches_reference.count())


In [0]:
# Inspeccionar el schema bronze accounts
df_accounts.printSchema()


In [0]:

# Inspeccionar algunos registros
display(df_accounts.limit(10))


In [0]:
# NORMALIZACION y CONVERSION DE TIPOS


df_accounts_base = (
    df_accounts

    # Normalización de identificadores
    .withColumn(
        "account_id",
        F.when(
            F.trim(F.col("account_id")) == "",
            None
        ).otherwise(F.upper(F.trim(F.col("account_id"))))
    )
    .withColumn(
        "customer_id",
        F.when(
            F.trim(F.col("customer_id")) == "",
            None
        ).otherwise(F.upper(F.trim(F.col("customer_id"))))
    )
    .withColumn(
        "branch_id",
        F.when(
            F.trim(F.col("branch_id")) == "",
            None
        ).otherwise(F.upper(F.trim(F.col("branch_id"))))
    )

    # Normalización de atributos
    .withColumn(
        "account_type",
        F.upper(F.trim(F.col("account_type")))
    )
    .withColumn(
        "currency",
        F.upper(F.trim(F.col("currency")))
    )
    .withColumn(
        "account_status",
        F.upper(F.trim(F.col("account_status")))
    )

    # Conversión de balance a decimal
    .withColumn(
        "balance",
        F.expr("try_cast(balance AS DECIMAL(18,2))")
    )

    # Conversión de fecha
    .withColumn(
        "opening_date",
        F.expr("try_cast(opening_date AS DATE)")
    )
)

display(df_accounts_base)


In [0]:

# Revisar los resultados
df_accounts_base.printSchema()

display(
    df_accounts_base.select(
        "account_id",
        "customer_id",
        "branch_id",
        "account_type",
        "balance",
        "currency",
        "opening_date",
        "account_status"
    ).limit(10)
)


In [0]:
from pyspark.sql.window import Window
# Identificar cuentas duplicadas

window_account = (
    Window
    .partitionBy("account_id")
    .orderBy(
        F.col("_ingestion_timestamp").desc_nulls_last(),
        F.col("_source_file").asc()
    )
)

df_accounts_dedup = (
    df_accounts_base
    .withColumn(
        "_row_number",
        F.row_number().over(window_account)
    )
    .withColumn(
        "_is_duplicate",
        F.when(
            F.col("account_id").isNotNull()
            & (F.col("_row_number") > 1),
            True
        ).otherwise(False)
    )
)

print("Accounts después de identificar duplicados:", df_accounts_dedup.count())

display(
    df_accounts_dedup
    .filter(F.col("_is_duplicate") == True)
)


In [0]:
# Integridad referencial
# Referencias de clientes válidos
df_customer_ids = (
    df_customers_reference
    .select("customer_id")
    .withColumn("_customer_exists", F.lit(True))
)

# Referencias de agencias válidas
df_branch_ids = (
    df_branches_reference
    .select("branch_id")
    .withColumn("_branch_exists", F.lit(True))
)


In [0]:
# Cruzar account con ambas referencias (branches y customers)

df_accounts_checked = (
    df_accounts_dedup.alias("a")

    .join(
        df_customer_ids.alias("c"),
        F.col("a.customer_id") == F.col("c.customer_id"),
        "left"
    )

    .join(
        df_branch_ids.alias("b"),
        F.col("a.branch_id") == F.col("b.branch_id"),
        "left"
    )

    .select(
        "a.*",
        F.coalesce(
            F.col("c._customer_exists"),
            F.lit(False)
        ).alias("_customer_exists"),
        F.coalesce(
            F.col("b._branch_exists"),
            F.lit(False)
        ).alias("_branch_exists")
    )
)

display(
    df_accounts_checked.select(
        "account_id",
        "customer_id",
        "_customer_exists",
        "branch_id",
        "_branch_exists"
    ).limit(20)
)


In [0]:
# Reglas de calidad

valid_account_types = [
    "SAVINGS",
    "CHECKING",
    "FIXED_TERM"
]

valid_currencies = [
    "PEN",
    "USD"
]

valid_account_status = [
    "ACTIVE",
    "INACTIVE",
    "BLOCKED"
]

df_accounts_quality = (
    df_accounts_checked
    .withColumn(
        "_rejection_reason",
        F.concat_ws(
            ", ",

            F.when(
                F.col("account_id").isNull(),
                "MISSING_ACCOUNT_ID"
            ),

            F.when(
                F.col("customer_id").isNull(),
                "MISSING_CUSTOMER_ID"
            ),

            F.when(
                F.col("customer_id").isNotNull()
                & (~F.col("_customer_exists")),
                "INVALID_CUSTOMER_REFERENCE"
            ),

            F.when(
                F.col("branch_id").isNull(),
                "MISSING_BRANCH_ID"
            ),

            F.when(
                F.col("branch_id").isNotNull()
                & (~F.col("_branch_exists")),
                "INVALID_BRANCH_REFERENCE"
            ),

            F.when(
                F.col("account_type").isNull()
                | (~F.col("account_type").isin(valid_account_types)),
                "INVALID_ACCOUNT_TYPE"
            ),

            F.when(
                F.col("balance").isNull(),
                "INVALID_BALANCE"
            ),

            F.when(
                F.col("balance") < 0,
                "NEGATIVE_BALANCE"
            ),

            F.when(
                F.col("currency").isNull()
                | (~F.col("currency").isin(valid_currencies)),
                "INVALID_CURRENCY"
            ),

            F.when(
                F.col("opening_date").isNull()
                | (F.col("opening_date") > F.current_date()),
                "INVALID_OPENING_DATE"
            ),

            F.when(
                F.col("account_status").isNull()
                | (~F.col("account_status").isin(valid_account_status)),
                "INVALID_ACCOUNT_STATUS"
            ),

            F.when(
                F.col("_is_duplicate"),
                "DUPLICATE_ACCOUNT_ID"
            )
        )
    )
)

display(
    df_accounts_quality
    .filter(F.col("_rejection_reason") != "")
    .select(
        "account_id",
        "customer_id",
        "branch_id",
        "balance",
        "account_type",
        "_is_duplicate",
        "_rejection_reason"
    )
)


In [0]:

# Revisar resultados de Accounts
total_bronze_accounts = df_accounts.count()

total_silver_accounts = (
    df_accounts_quality
    .filter(F.col("_rejection_reason") == "")
    .count()
)

total_quarantine_accounts = (
    df_accounts_quality
    .filter(F.col("_rejection_reason") != "")
    .count()
)

print("Total Bronze:", total_bronze_accounts)
print("Total Silver esperado:", total_silver_accounts)
print("Total Quarantine esperado:", total_quarantine_accounts)
print(
    "Total procesado:",
    total_silver_accounts + total_quarantine_accounts
)

display(
    df_accounts_quality
    .filter(F.col("_rejection_reason") != "")
    .groupBy("_rejection_reason")
    .count()
    .orderBy(F.desc("count"))
)


In [0]:
# Guardando las tablas a Capa Silver

# DataFrame Silver: cuentas válidas
df_silver_accounts = (
    df_accounts_quality
    .filter(F.col("_rejection_reason") == "")
    .drop(
        "_row_number",
        "_is_duplicate",
        "_customer_exists",
        "_branch_exists",
        "_rejection_reason"
    )
)

# DataFrame Quarantine: cuentas rechazadas
df_quarantine_accounts = (
    df_accounts_quality
    .filter(F.col("_rejection_reason") != "")
    .drop("_row_number")
)

# Guardar Silver
(
    df_silver_accounts.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLES["accounts"])
)

# Guardar Quarantine
(
    df_quarantine_accounts.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.{SCHEMA}.quarantine_accounts"
    )
)

print("Tablas Accounts creadas correctamente.")


In [0]:
# Validaciones
df_silver_accounts_saved = spark.table(
    SILVER_TABLES["accounts"]
)

df_quarantine_accounts_saved = spark.table(
    f"{CATALOG}.{SCHEMA}.quarantine_accounts"
)

count_bronze = df_accounts.count()
count_silver = df_silver_accounts_saved.count()
count_quarantine = df_quarantine_accounts_saved.count()

print("Bronze:", count_bronze)
print("Silver:", count_silver)
print("Quarantine:", count_quarantine)
print("Total reconciliado:", count_silver + count_quarantine)

assert count_bronze == count_silver + count_quarantine

print("Validación de reconciliación exitosa.")


In [0]:
# Leer Transactions desde Bronze
df_transactions = spark.table(
    BRONZE_TABLES["transactions"]
)

# Leer Accounts desde Silver
df_accounts_reference = spark.table(
    SILVER_TABLES["accounts"]
)

print("Total Transactions Bronze:", df_transactions.count())
print("Total Accounts Silver:", df_accounts_reference.count())


In [0]:
# Revisar el schema y algunos datos de muestra
df_transactions.printSchema()

display(df_transactions.limit(10))


In [0]:

# Normalizar y conversión de datos

df_transactions_base = (
    df_transactions

    # Normalización de identificadores
    .withColumn(
        "transaction_id",
        F.when(
            F.trim(F.col("transaction_id")) == "",
            None
        ).otherwise(F.upper(F.trim(F.col("transaction_id"))))
    )
    .withColumn(
        "account_id",
        F.when(
            F.trim(F.col("account_id")) == "",
            None
        ).otherwise(F.upper(F.trim(F.col("account_id"))))
    )

    # Normalización de atributos
    .withColumn(
        "channel",
        F.upper(F.trim(F.col("channel")))
    )
    .withColumn(
        "currency",
        F.upper(F.trim(F.col("currency")))
    )
    .withColumn(
        "status",
        F.upper(F.trim(F.col("status")))
    )
    .withColumn(
        "transaction_type",
        F.upper(F.trim(F.col("transaction_type")))
    )
    .withColumn(
        "description",
        F.trim(F.col("description"))
    )

    # Conversión de importe financiero
    .withColumn(
        "amount",
        F.expr("try_cast(amount AS DECIMAL(18,2))")
    )

    # Conversión de fecha
    .withColumn(
        "transaction_date",
        F.expr("try_cast(transaction_date AS TIMESTAMP)")
    )
)

display(df_transactions_base)


In [0]:

df_transactions_base.printSchema()


In [0]:

# Indetificación de Transacciones duplicadas

from pyspark.sql.window import Window

window_transaction = (
    Window
    .partitionBy("transaction_id")
    .orderBy(
        F.col("_ingestion_timestamp").desc_nulls_last(),
        F.col("_source_file").asc()
    )
)

df_transactions_dedup = (
    df_transactions_base
    .withColumn(
        "_row_number",
        F.row_number().over(window_transaction)
    )
    .withColumn(
        "_is_duplicate",
        F.when(
            F.col("transaction_id").isNotNull()
            & (F.col("_row_number") > 1),
            True
        ).otherwise(False)
    )
)

print(
    "Registros identificados:",
    df_transactions_dedup.count()
)

display(
    df_transactions_dedup
    .filter(F.col("_is_duplicate") == True)
)


In [0]:
# Validar integridad referencial
df_account_ids = (
    df_accounts_reference
    .select("account_id")
    .withColumn("_account_exists", F.lit(True))
)


df_transactions_checked = (
    df_transactions_dedup.alias("t")

    .join(
        df_account_ids.alias("a"),
        F.col("t.account_id") == F.col("a.account_id"),
        "left"
    )

    .select(
        "t.*",
        F.coalesce(
            F.col("a._account_exists"),
            F.lit(False)
        ).alias("_account_exists")
    )
)

display(
    df_transactions_checked.select(
        "transaction_id",
        "account_id",
        "_account_exists"
    ).limit(20)
)


In [0]:
# APLICAR REGLAS DE CALIDAD

valid_transaction_types = [
    "DEPOSIT",
    "WITHDRAWAL",
    "TRANSFER",
    "PAYMENT"
]

valid_currencies = [
    "PEN",
    "USD"
]

valid_statuses = [
    "COMPLETED",
    "PENDING",
    "REJECTED"
]

valid_channels = [
    "MOBILE_APP",
    "WEB",
    "ATM",
    "BRANCH",
    "POS"
]


df_transactions_quality = (
    df_transactions_checked
    .withColumn(
        "_rejection_reason",
        F.concat_ws(
            ", ",

            F.when(
                F.col("transaction_id").isNull(),
                "MISSING_TRANSACTION_ID"
            ),

            F.when(
                F.col("account_id").isNull(),
                "MISSING_ACCOUNT_ID"
            ),

            F.when(
                F.col("account_id").isNotNull()
                & (~F.col("_account_exists")),
                "INVALID_ACCOUNT_REFERENCE"
            ),

            F.when(
                F.col("amount").isNull(),
                "INVALID_AMOUNT"
            ),

            F.when(
                F.col("amount") <= 0,
                "NON_POSITIVE_AMOUNT"
            ),

            F.when(
                F.col("transaction_date").isNull(),
                "INVALID_TRANSACTION_DATE"
            ),

            F.when(
                F.col("transaction_type").isNull()
                | (~F.col("transaction_type").isin(valid_transaction_types)),
                "INVALID_TRANSACTION_TYPE"
            ),

            F.when(
                F.col("currency").isNull()
                | (~F.col("currency").isin(valid_currencies)),
                "INVALID_CURRENCY"
            ),

            F.when(
                F.col("status").isNull()
                | (~F.col("status").isin(valid_statuses)),
                "INVALID_STATUS"
            ),

            F.when(
                F.col("channel").isNull()
                | (~F.col("channel").isin(valid_channels)),
                "INVALID_CHANNEL"
            ),

            F.when(
                F.col("_is_duplicate"),
                "DUPLICATE_TRANSACTION_ID"
            )
        )
    )
)

display(
    df_transactions_quality
    .filter(F.col("_rejection_reason") != "")
    .select(
        "transaction_id",
        "account_id",
        "amount",
        "transaction_date",
        "transaction_type",
        "status",
        "channel",
        "_rejection_reason"
    )
)


In [0]:
# Validación de resultados

total_bronze_transactions = df_transactions.count()

total_silver_transactions = (
    df_transactions_quality
    .filter(F.col("_rejection_reason") == "")
    .count()
)

total_quarantine_transactions = (
    df_transactions_quality
    .filter(F.col("_rejection_reason") != "")
    .count()
)

print("Total Bronze:", total_bronze_transactions)
print("Total Silver esperado:", total_silver_transactions)
print("Total Quarantine esperado:", total_quarantine_transactions)

print(
    "Total procesado:",
    total_silver_transactions + total_quarantine_transactions
)

display(
    df_transactions_quality
    .filter(F.col("_rejection_reason") != "")
    .groupBy("_rejection_reason")
    .count()
    .orderBy(F.desc("count"))
)


In [0]:

# DataFrame Silver: transacciones válidas
df_silver_transactions = (
    df_transactions_quality
    .filter(F.col("_rejection_reason") == "")
    .drop(
        "_row_number",
        "_is_duplicate",
        "_account_exists",
        "_rejection_reason"
    )
)

# DataFrame Quarantine: transacciones rechazadas
df_quarantine_transactions = (
    df_transactions_quality
    .filter(F.col("_rejection_reason") != "")
    .drop("_row_number")
)

# Guardar Silver
(
    df_silver_transactions.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLES["transactions"])
)

# Guardar Quarantine
(
    df_quarantine_transactions.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.{SCHEMA}.quarantine_transactions"
    )
)

print("Tablas Transactions creadas correctamente.")


In [0]:

# Validcion
df_silver_transactions_saved = spark.table(
    SILVER_TABLES["transactions"]
)

df_quarantine_transactions_saved = spark.table(
    f"{CATALOG}.{SCHEMA}.quarantine_transactions"
)

count_bronze = df_transactions.count()
count_silver = df_silver_transactions_saved.count()
count_quarantine = df_quarantine_transactions_saved.count()

print("Bronze:", count_bronze)
print("Silver:", count_silver)
print("Quarantine:", count_quarantine)
print("Total reconciliado:", count_silver + count_quarantine)

assert count_bronze == count_silver + count_quarantine

print("Validación de reconciliación exitosa.")
